# 02. Data Preprocessing & Leakage Prevention Pipeline

**Objective:** Clean raw job description text, construct the binary target, perform stratified train/val/test splits, and build the token vocabulary strictly on the training partition.

---

## 1. Preprocessing Architecture
To ensure scientific validity and zero test-data leakage:
1. Filter only rows with valid `formatted_experience_level`.
2. Clean text: strip HTML markup, URLs, special characters, normalize whitespace, lowercase.
3. Split data: Stratified 65% Train, 15% Validation, 20% Holdout Test.
4. Fit `TextVocabulary` exclusively on the training split.
5. Encode and pad sequences to `MAX_SEQ_LENGTH = 250`.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append("..")
from src import config, preprocessing

print("Preprocessing module loaded.")


Preprocessing module loaded.


## 2. Text Cleaning Demonstration
Raw job postings often contain HTML formatting tags (`<p>`, `<li>`, `<br>`) and URL links.


In [2]:
sample_raw_text = """<p>We are seeking a <b>Senior Python Engineer</b> with expertise in PyTorch and AWS! Visit https://example.com/careers for more info.</p>"""
cleaned = preprocessing.clean_text(sample_raw_text)
print("Raw Input:     ", sample_raw_text)
print("Cleaned Output:", cleaned)


Raw Input:      <p>We are seeking a <b>Senior Python Engineer</b> with expertise in PyTorch and AWS! Visit https://example.com/careers for more info.</p>
Cleaned Output: we are seeking a senior python engineer with expertise in pytorch and aws visit for more info


## 3. Vocabulary Fitting & Zero-Leakage Guarantee
The vocabulary maps words to unique integer indices. Word indices $0$ and $1$ are reserved for `<PAD>` and `<UNK>` tokens respectively. The vocabulary is fitted strictly on `train_df['description']`.


In [3]:
vocab = preprocessing.TextVocabulary(max_vocab_size=config.MAX_VOCAB_SIZE, min_freq=config.MIN_WORD_FREQ)
sample_corpus = [
    "python deep learning pytorch cnn model",
    "market signal job posting requirement startup",
    "software engineering agile sprint delivery backlog"
]
vocab.fit(sample_corpus)
print(f"Vocab size: {len(vocab)}")
encoded = vocab.encode("python startup model deployment", max_length=10)
print("Encoded Sequence:", encoded)


Vocab size: 2
Encoded Sequence: [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]


## 4. Reusable Preprocessing Pipeline Summary
The full pipeline ensures modular execution via `src/preprocessing.py`, generating preprocessed partitions ready for model training without data leakage.
